[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/solutions/b_28_kv_cache_pure_solution.ipynb)

# 🔴 Solution: KV Cache Attention without Flax

*Attention & Transformers · Hard*

Reference implementation. Try it yourself in `b_28_kv_cache_pure.ipynb` first.

---
Problem 14 with an explicit parameter pytree — and a cache that was always a
value, not state.

### Signature
```python
def init_kv_attention(key, d_model, num_heads):
    ...   # -> the same four-projection pytree as b_26

def apply_kv_attention(params, x, num_heads, cache=None):
    ...   # -> (out, (k_all, v_all))
```

| | shape |
|---|---|
| `x` | `(B, seq_new, d_model)` |
| `cache` | `None`, or `(k, v)` each `(B, H, seq_past, d_k)` |
| `out` | `(B, seq_new, d_model)` |
| returned cache | `(k, v)` each `(B, H, seq_past + seq_new, d_k)` |

### Two things are now explicit that a module hid
**The parameters.** Same pytree as `b_26`: four `(d_model, d_model)` kernels,
key split four ways.

**The cache.** In problem 14 it already had to be returned rather than mutated,
because JAX arrays are immutable — so `nnx.Module` was buying you nothing there.
Written as a plain function that becomes obvious: the cache goes in as an
argument and comes back as a return value, exactly like the parameters.

### The mask is the hard part, and it is silent
Query `i` of this chunk sits at absolute position `seq_past + i`, so it may
attend to keys `0 … seq_past + i`:

$$j - i \le \text{seq\_past} \quad\Longrightarrow\quad
\texttt{tril(ones((seq\_new, seq\_total)), k=seq\_total - seq\_new)}$$

Forget the `k=` and you get the top-left triangle, which hides every cached
key. Nothing errors — the model just stops seeing its own history.

During single-token decode `seq_new == 1`, the mask is all-True and does
nothing; the bug only shows up when you prefill more than one token at a time.

### The test that catches everything
Run a sequence two ways — all at once, versus prefill-then-decode-one-at-a-time —
and require they agree. Wrong mask offset, wrong concat axis, or re-projecting
the cache all fail it.

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✅ REFERENCE SOLUTION

import jax
import jax.numpy as jnp


def init_kv_attention(key, d_model, num_heads):
    keys = jax.random.split(key, 4)
    return {
        name: {
            "kernel": jax.random.normal(k, (d_model, d_model)) / jnp.sqrt(d_model),
            "bias": jnp.zeros((d_model,)),
        }
        for name, k in zip(("W_q", "W_k", "W_v", "W_o"), keys)
    }


def _dense(p, x):
    return x @ p["kernel"] + p["bias"]


def apply_kv_attention(params, x, num_heads, cache=None):
    d_model = x.shape[-1]
    d_k = d_model // num_heads

    def heads(t):
        return t.reshape(*t.shape[:-1], num_heads, d_k).swapaxes(-3, -2)

    q = heads(_dense(params["W_q"], x))
    k = heads(_dense(params["W_k"], x))
    v = heads(_dense(params["W_v"], x))

    # The cache is a value we extend, not state we mutate.
    if cache is not None:
        k = jnp.concatenate([cache[0], k], axis=-2)
        v = jnp.concatenate([cache[1], v], axis=-2)
    new_cache = (k, v)

    seq_new, seq_total = q.shape[-2], k.shape[-2]
    scores = jnp.einsum("...hqd,...hkd->...hqk", q, k) / jnp.sqrt(
        jnp.asarray(d_k, x.dtype)
    )
    # Query i is at absolute position seq_past + i, so allowed is
    # j - i <= seq_past. Plain tril would hide the entire cache.
    allowed = jnp.tril(
        jnp.ones((seq_new, seq_total), dtype=bool), k=seq_total - seq_new
    )
    scores = jnp.where(allowed, scores, -jnp.inf)

    o = jnp.einsum("...hqk,...hkd->...hqd", jax.nn.softmax(scores, axis=-1), v)
    o = o.swapaxes(-3, -2)
    o = o.reshape(*o.shape[:-2], d_model)
    return _dense(params["W_o"], o), new_cache

In [ ]:
# 🔍 Verify
import jax
import jax.numpy as jnp

params = init_kv_attention(jax.random.key(0), d_model=8, num_heads=2)
x = jax.random.normal(jax.random.key(1), (1, 8, 8))

full, _ = apply_kv_attention(params, x, 2)

out, cache = apply_kv_attention(params, x[:, :5], 2)
print("prefill 5:", out.shape, "cache:", cache[0].shape)
pieces = [out]
for t in range(5, 8):
    out, cache = apply_kv_attention(params, x[:, t:t + 1], 2, cache)
    print(f"  decode {t}: cache grew to {cache[0].shape[-2]}")
    pieces.append(out)

step = jnp.concatenate(pieces, axis=1)
print("\nstepwise == all-at-once?", bool(jnp.allclose(step, full, atol=1e-5)),
      f"(max diff {float(jnp.max(jnp.abs(step - full))):.2e})")

In [ ]:
# Run the judge against the reference solution
from jax_judge import check

check("kv_cache_pure")